In [0]:
telecom_bronze_df = spark.table(
    "workspace.census360.bronze_cbsl_telecommunication_services_raw"
)

print("Row count:", telecom_bronze_df.count())
print("\nColumns:")
print(telecom_bronze_df.columns)

print("\nSchema:")
telecom_bronze_df.printSchema()

print("\nRaw telecommunications rows:")
display(
    telecom_bronze_df
    .orderBy("source_row_number")
)

In [0]:
display(
    telecom_bronze_df
    .orderBy("source_row_number")
    .select(
        "source_row_number",
        "column_00",
        "column_01",
        "column_02",
        "column_03",
        "column_04",
        "column_05",
        "column_06",
        "column_07",
        "column_08",
        "column_09",
        "column_10",
        "column_11",
        "column_12",
        "column_13"
    )
)

In [0]:
from pyspark.sql import functions as F

telecom_classified_df = (
    telecom_bronze_df
    .withColumn(
        "row_type",
        F.when(F.col("source_row_number").between(1, 2), "title")
         .when(F.col("source_row_number") == 3, "year_header")
         .when(F.col("source_row_number").isin(4, 10), "section_heading")
         .when(F.col("source_row_number").between(5, 9), "metric")
         .when(F.col("source_row_number").between(11, 14), "metric")
         .when(F.col("source_row_number").between(15, 16), "blank")
         .when(F.col("source_row_number").between(17, 21), "footnote")
         .when(F.col("source_row_number") == 22, "blank")
         .when(F.col("source_row_number").between(23, 26), "source")
         .otherwise("unclassified")
    )
)

display(
    telecom_classified_df
    .orderBy("source_row_number")
    .select(
        "source_row_number",
        "row_type",
        "column_00",
        "column_01"
    )
)

In [0]:
from pyspark.sql import functions as F

telecom_long_df = (
    telecom_bronze_df
    .filter(F.col("source_row_number").isin(5, 6, 7, 8, 9, 11, 12, 13, 14))

    # Preserve the worksheet section context.
    .withColumn(
        "section_name",
        F.when(
            F.col("source_row_number").between(5, 9),
            F.lit("Fixed Access Services")
        ).otherwise(F.lit("Other Services"))
    )

    # Add stable analytical metric names.
    .withColumn(
        "metric_code",
        F.when(F.col("source_row_number") == 5, "fixed_subscriber_base_thousand")
         .when(F.col("source_row_number") == 6, "wireline_telephones_thousand")
         .when(F.col("source_row_number") == 7, "wireless_local_loop_thousand")
         .when(F.col("source_row_number") == 8, "fixed_subscriber_growth_percent")
         .when(F.col("source_row_number") == 9, "telephone_penetration_per_100")
         .when(F.col("source_row_number") == 11, "cellular_phones_thousand")
         .when(F.col("source_row_number") == 12, "cellular_penetration_per_100")
         .when(F.col("source_row_number") == 13, "public_pay_phone_booths")
         .when(F.col("source_row_number") == 14, "internet_connections_total")
    )

    # Convert the year columns into rows.
    .withColumn(
        "year_values",
        F.array(
            F.struct(F.lit(2013).alias("year"), F.col("column_02").alias("value_raw")),
            F.struct(F.lit(2014).alias("year"), F.col("column_03").alias("value_raw")),
            F.struct(F.lit(2015).alias("year"), F.col("column_04").alias("value_raw")),
            F.struct(F.lit(2016).alias("year"), F.col("column_05").alias("value_raw")),
            F.struct(F.lit(2017).alias("year"), F.col("column_06").alias("value_raw")),
            F.struct(F.lit(2018).alias("year"), F.col("column_07").alias("value_raw")),
            F.struct(F.lit(2019).alias("year"), F.col("column_08").alias("value_raw")),
            F.struct(F.lit(2020).alias("year"), F.col("column_09").alias("value_raw")),
            F.struct(F.lit(2021).alias("year"), F.col("column_10").alias("value_raw")),
            F.struct(F.lit(2022).alias("year"), F.col("column_11").alias("value_raw")),
            F.struct(F.lit(2023).alias("year"), F.col("column_12").alias("value_raw")),
            F.struct(F.lit(2024).alias("year"), F.col("column_13").alias("value_raw"))
        )
    )
    .withColumn("year_value", F.explode("year_values"))

    .select(
        "source_row_number",
        "section_name",
        "metric_code",
        F.col("column_01").alias("metric_label"),
        F.col("year_value.year").alias("year"),
        F.col("year_value.value_raw").alias("value_raw"),
        "source_file",
        "source_sheet"
    )

    # Convert valid numbers while preserving values such as n.a.
    .withColumn(
        "value_numeric",
        F.when(
            F.trim(F.col("value_raw")).rlike(r"^-?\d+(\.\d+)?$"),
            F.trim(F.col("value_raw")).cast("double")
        )
    )

    # Preserve the official revision status.
    .withColumn(
        "data_status",
        F.when(F.col("year") == 2023, F.lit("revised"))
         .when(F.col("year") == 2024, F.lit("provisional"))
         .otherwise(F.lit("published"))
    )
)

display(
    telecom_long_df.orderBy("source_row_number", "year")
)

In [0]:
from pyspark.sql import functions as F

telecom_validation_df = (
    telecom_long_df
    .groupBy("metric_code")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("year").alias("year_count"),
        F.min("year").alias("first_year"),
        F.max("year").alias("last_year"),
        F.sum(
            F.when(F.col("value_numeric").isNull(), 1).otherwise(0)
        ).alias("null_numeric_count")
    )
    .orderBy("metric_code")
)

display(telecom_validation_df)

In [0]:
silver_telecom_table = (
    "workspace.census360.silver_cbsl_telecommunication_services"
)

table_exists = spark.catalog.tableExists(silver_telecom_table)

print("Target table:", silver_telecom_table)
print("Already exists:", table_exists)

if table_exists:
    existing_df = spark.table(silver_telecom_table)
    print("Existing row count:", existing_df.count())
    print("Existing columns:", existing_df.columns)

In [0]:
from pyspark.sql import functions as F

telecom_silver_df = (
    telecom_long_df
    .withColumn(
        "value_status",
        F.when(
            F.col("value_numeric").isNull(),
            F.lit("not_available")
        ).otherwise(F.lit("available"))
    )
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
    .select(
        "year",
        "section_name",
        "metric_code",
        "metric_label",
        "value_raw",
        "value_numeric",
        "value_status",
        "data_status",
        "source_row_number",
        "source_file",
        "source_sheet",
        "silver_processed_at"
    )
)

# Temporary view used only for creating the permanent Delta table.
telecom_silver_df.createOrReplaceTempView(
    "temp_cbsl_telecommunication_services"
)

spark.sql("""
    CREATE TABLE workspace.census360.silver_cbsl_telecommunication_services
    USING DELTA
    AS
    SELECT *
    FROM temp_cbsl_telecommunication_services
""")

created_df = spark.table(
    "workspace.census360.silver_cbsl_telecommunication_services"
)

print("Silver telecom table created successfully.")
print("Rows:", created_df.count())
print("Columns:", len(created_df.columns))

In [0]:
hies_bronze_df = spark.table(
    "workspace.census360.bronze_cbsl_household_income_hies_raw"
)

print("Row count:", hies_bronze_df.count())
print("\nColumns:")
print(hies_bronze_df.columns)

print("\nSchema:")
hies_bronze_df.printSchema()

print("\nRaw HIES rows:")
display(
    hies_bronze_df
    .orderBy("source_row_number")
)

In [0]:
from pyspark.sql import functions as F

hies_classified_df = (
    hies_bronze_df
    .withColumn(
        "row_type",
        F.when(F.col("source_row_number").between(1, 2), "title")
         .when(F.col("source_row_number") == 3, "column_header")
         .when(F.col("source_row_number") == 4, "survey_period_header")
         .when(F.col("source_row_number").between(5, 21), "metric")
         .when(F.col("source_row_number") == 22, "blank")
         .when(F.col("source_row_number") == 23, "source")
         .when(F.col("source_row_number") == 24, "note")
         .otherwise("unclassified")
    )
)

display(
    hies_classified_df
    .orderBy("source_row_number")
    .select(
        "source_row_number",
        "row_type",
        "column_01",
        "column_02",
        "column_03",
        "column_12"
    )
)

In [0]:
from pyspark.sql import functions as F

hies_long_df = (
    hies_bronze_df
    .filter(F.col("source_row_number").between(5, 21))

    # Assign stable analytical metric names.
    .withColumn(
        "metric_code",
        F.when(F.col("source_row_number") == 5, "mean_household_income_monthly_lkr")
         .when(F.col("source_row_number") == 6, "median_household_income_monthly_lkr")
         .when(F.col("source_row_number") == 7, "mean_per_capita_income_monthly_lkr")
         .when(F.col("source_row_number") == 8, "income_receiver_mean_income_monthly_lkr")
         .when(F.col("source_row_number") == 9, "income_receivers_per_household")
         .when(F.col("source_row_number") == 10, "household_size")
         .when(F.col("source_row_number") == 11, "monetary_income_household_monthly_lkr")
         .when(F.col("source_row_number") == 12, "non_monetary_income_household_monthly_lkr")
         .when(F.col("source_row_number") == 13, "gini_household_income")
         .when(F.col("source_row_number") == 14, "gini_household_expenditure")
         .when(F.col("source_row_number") == 15, "gini_income_receiver_income")
         .when(F.col("source_row_number") == 16, "mean_household_expenditure_monthly_lkr")
         .when(F.col("source_row_number") == 17, "food_drink_expenditure_monthly_lkr")
         .when(F.col("source_row_number") == 18, "non_food_expenditure_monthly_lkr")
         .when(F.col("source_row_number") == 19, "liquor_narcotics_tobacco_expenditure_monthly_lkr")
         .when(F.col("source_row_number") == 20, "food_ratio_percent")
         .when(F.col("source_row_number") == 21, "poverty_headcount_ratio_2002_based_percent")
    )

    # Convert the irregular survey-period columns into rows.
    .withColumn(
        "survey_values",
        F.array(
            F.struct(
                F.lit("1985/86").alias("survey_period"),
                F.lit(1985).alias("period_start_year"),
                F.lit(1986).alias("period_end_year"),
                F.col("column_03").alias("value_raw")
            ),
            F.struct(
                F.lit("1990/91").alias("survey_period"),
                F.lit(1990).alias("period_start_year"),
                F.lit(1991).alias("period_end_year"),
                F.col("column_04").alias("value_raw")
            ),
            F.struct(
                F.lit("1995/96").alias("survey_period"),
                F.lit(1995).alias("period_start_year"),
                F.lit(1996).alias("period_end_year"),
                F.col("column_05").alias("value_raw")
            ),
            F.struct(
                F.lit("2002").alias("survey_period"),
                F.lit(2002).alias("period_start_year"),
                F.lit(2002).alias("period_end_year"),
                F.col("column_06").alias("value_raw")
            ),
            F.struct(
                F.lit("2005").alias("survey_period"),
                F.lit(2005).alias("period_start_year"),
                F.lit(2005).alias("period_end_year"),
                F.col("column_07").alias("value_raw")
            ),
            F.struct(
                F.lit("2006/07").alias("survey_period"),
                F.lit(2006).alias("period_start_year"),
                F.lit(2007).alias("period_end_year"),
                F.col("column_08").alias("value_raw")
            ),
            F.struct(
                F.lit("2009/10").alias("survey_period"),
                F.lit(2009).alias("period_start_year"),
                F.lit(2010).alias("period_end_year"),
                F.col("column_09").alias("value_raw")
            ),
            F.struct(
                F.lit("2012/13").alias("survey_period"),
                F.lit(2012).alias("period_start_year"),
                F.lit(2013).alias("period_end_year"),
                F.col("column_10").alias("value_raw")
            ),
            F.struct(
                F.lit("2016").alias("survey_period"),
                F.lit(2016).alias("period_start_year"),
                F.lit(2016).alias("period_end_year"),
                F.col("column_11").alias("value_raw")
            ),
            F.struct(
                F.lit("2019").alias("survey_period"),
                F.lit(2019).alias("period_start_year"),
                F.lit(2019).alias("period_end_year"),
                F.col("column_12").alias("value_raw")
            )
        )
    )
    .withColumn("survey_value", F.explode("survey_values"))

    .select(
        "source_row_number",
        "metric_code",
        F.col("column_01").alias("metric_label"),
        F.col("column_02").alias("unit"),
        F.col("survey_value.survey_period").alias("survey_period"),
        F.col("survey_value.period_start_year").alias("period_start_year"),
        F.col("survey_value.period_end_year").alias("period_end_year"),
        F.col("survey_value.value_raw").alias("value_raw"),
        "source_file",
        "source_sheet"
    )

    # Convert valid numeric strings while preserving unavailable symbols.
    .withColumn(
        "value_clean",
        F.regexp_replace(
            F.trim(F.col("value_raw")),
            ",",
            ""
        )
    )
    .withColumn(
        "value_numeric",
        F.when(
            F.col("value_clean").rlike(r"^-?\d+(\.\d+)?$"),
            F.col("value_clean").cast("double")
        )
    )
    .drop("value_clean")
)

print("HIES long-format rows:", hies_long_df.count())

display(
    hies_long_df.orderBy(
        "source_row_number",
        "period_start_year"
    )
)

In [0]:
from pyspark.sql import functions as F

hies_validation_df = (
    hies_long_df
    .groupBy("metric_code")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("survey_period").alias("survey_period_count"),
        F.min("period_start_year").alias("earliest_start_year"),
        F.max("period_end_year").alias("latest_end_year"),
        F.sum(
            F.when(F.col("value_numeric").isNull(), 1).otherwise(0)
        ).alias("null_numeric_count")
    )
    .orderBy("metric_code")
)

duplicate_count = (
    hies_long_df
    .groupBy("metric_code", "survey_period")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total rows:", hies_long_df.count())
print("Distinct metrics:", hies_long_df.select("metric_code").distinct().count())
print("Distinct survey periods:", hies_long_df.select("survey_period").distinct().count())
print("Duplicate metric-period combinations:", duplicate_count)

display(hies_validation_df)

In [0]:
silver_hies_table = (
    "workspace.census360.silver_cbsl_household_income_hies"
)

table_exists = spark.catalog.tableExists(silver_hies_table)

print("Target table:", silver_hies_table)
print("Already exists:", table_exists)

if table_exists:
    existing_hies_df = spark.table(silver_hies_table)
    print("Existing row count:", existing_hies_df.count())
    print("Existing columns:", existing_hies_df.columns)

In [0]:
from pyspark.sql import functions as F

hies_silver_df = (
    hies_long_df
    .withColumn(
        "value_status",
        F.when(
            F.col("value_numeric").isNull(),
            F.lit("not_available")
        ).otherwise(F.lit("available"))
    )
    .withColumn(
        "survey_period_type",
        F.when(
            F.col("period_start_year") == F.col("period_end_year"),
            F.lit("single_year")
        ).otherwise(F.lit("multi_year"))
    )
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
    .select(
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "metric_code",
        "metric_label",
        "unit",
        "value_raw",
        "value_numeric",
        "value_status",
        "source_row_number",
        "source_file",
        "source_sheet",
        "silver_processed_at"
    )
)

hies_silver_df.createOrReplaceTempView(
    "temp_cbsl_household_income_hies"
)

spark.sql("""
    CREATE TABLE workspace.census360.silver_cbsl_household_income_hies
    USING DELTA
    AS
    SELECT *
    FROM temp_cbsl_household_income_hies
""")

created_hies_df = spark.table(
    "workspace.census360.silver_cbsl_household_income_hies"
)

print("Silver HIES table created successfully.")
print("Rows:", created_hies_df.count())
print("Columns:", len(created_hies_df.columns))

In [0]:
poverty_bronze_df = spark.table(
    "workspace.census360.bronze_cbsl_poverty_indicators_raw"
)

print("Row count:", poverty_bronze_df.count())
print("\nColumns:")
print(poverty_bronze_df.columns)

print("\nSchema:")
poverty_bronze_df.printSchema()

print("\nRaw poverty rows:")
display(
    poverty_bronze_df
    .orderBy("source_row_number")
)

In [0]:
from pyspark.sql import functions as F

poverty_classified_df = (
    poverty_bronze_df
    .withColumn(
        "row_type",
        F.when(F.col("source_row_number").between(1, 2), "title")
         .when(F.col("source_row_number") == 3, "metric_header")
         .when(F.col("source_row_number") == 4, "survey_period_header")
         .when(F.col("source_row_number") == 5, "data")
         .when(F.col("source_row_number").isin(6, 10, 20), "section_heading")
         .when(F.col("source_row_number").between(7, 9), "data")
         .when(F.col("source_row_number").between(11, 19), "data")
         .when(F.col("source_row_number").between(21, 45), "data")
         .when(F.col("source_row_number").isin(46, 49), "blank")
         .when(F.col("source_row_number").between(47, 48), "footnote")
         .when(F.col("source_row_number").between(50, 51), "source")
         .otherwise("unclassified")
    )
    .withColumn(
        "geography_type",
        F.when(F.col("source_row_number") == 5, "national")
         .when(F.col("source_row_number").between(7, 9), "sector")
         .when(F.col("source_row_number").between(11, 19), "province")
         .when(F.col("source_row_number").between(21, 45), "district")
    )
)

display(
    poverty_classified_df
    .orderBy("source_row_number")
    .select(
        "source_row_number",
        "row_type",
        "geography_type",
        F.col("column_01").alias("row_label")
    )
)

In [0]:
from pyspark.sql import functions as F

poverty_value_mappings = [
    # Poverty Headcount Index
    ("poverty_headcount_index_percent", "Poverty Headcount Index (%)",
     "2012/13", 2012, 2013, "standard", "column_02"),

    ("poverty_headcount_index_percent", "Poverty Headcount Index (%)",
     "2016", 2016, 2016, "standard", "column_03"),

    ("poverty_headcount_index_percent", "Poverty Headcount Index (%)",
     "2019(a)", 2019, 2019, "old_poverty_line_2002_ccpi", "column_04"),

    ("poverty_headcount_index_percent", "Poverty Headcount Index (%)",
     "2019(b)", 2019, 2019, "updated_poverty_line_2012_13_ncpi", "column_05"),

    # Poor Household Percentage
    ("poor_household_percentage", "Poor Household Percentage",
     "2012/13", 2012, 2013, "standard", "column_06"),

    ("poor_household_percentage", "Poor Household Percentage",
     "2016", 2016, 2016, "standard", "column_07"),

    ("poor_household_percentage", "Poor Household Percentage",
     "2019(a)", 2019, 2019, "old_poverty_line_2002_ccpi", "column_08"),

    ("poor_household_percentage", "Poor Household Percentage",
     "2019(b)", 2019, 2019, "updated_poverty_line_2012_13_ncpi", "column_09"),

    # Poverty Gap Index
    ("poverty_gap_index_percent", "Poverty Gap Index (%)",
     "2012/13", 2012, 2013, "standard", "column_10"),

    ("poverty_gap_index_percent", "Poverty Gap Index (%)",
     "2016", 2016, 2016, "standard", "column_11"),

    ("poverty_gap_index_percent", "Poverty Gap Index (%)",
     "2019(a)", 2019, 2019, "old_poverty_line_2002_ccpi", "column_12"),

    ("poverty_gap_index_percent", "Poverty Gap Index (%)",
     "2019(b)", 2019, 2019, "updated_poverty_line_2012_13_ncpi", "column_13")
]

poverty_structs = [
    F.struct(
        F.lit(metric_code).alias("metric_code"),
        F.lit(metric_label).alias("metric_label"),
        F.lit(survey_period).alias("survey_period"),
        F.lit(start_year).alias("period_start_year"),
        F.lit(end_year).alias("period_end_year"),
        F.lit(definition_variant).alias("definition_variant"),
        F.col(source_column).alias("value_raw")
    )
    for (
        metric_code,
        metric_label,
        survey_period,
        start_year,
        end_year,
        definition_variant,
        source_column
    ) in poverty_value_mappings
]

poverty_long_df = (
    poverty_classified_df
    .filter(F.col("row_type") == "data")
    .withColumn(
        "poverty_values",
        F.array(*poverty_structs)
    )
    .withColumn(
        "poverty_value",
        F.explode("poverty_values")
    )
    .select(
        "source_row_number",
        "geography_type",
        F.trim(F.col("column_01")).alias("geography_name"),
        F.col("poverty_value.metric_code").alias("metric_code"),
        F.col("poverty_value.metric_label").alias("metric_label"),
        F.col("poverty_value.survey_period").alias("survey_period"),
        F.col("poverty_value.period_start_year").alias("period_start_year"),
        F.col("poverty_value.period_end_year").alias("period_end_year"),
        F.col("poverty_value.definition_variant").alias("definition_variant"),
        F.col("poverty_value.value_raw").alias("value_raw"),
        "source_file",
        "source_sheet"
    )
    .withColumn(
        "value_clean",
        F.regexp_replace(
            F.trim(F.col("value_raw")),
            ",",
            ""
        )
    )
    .withColumn(
        "value_numeric",
        F.when(
            F.col("value_clean").rlike(r"^-?\d+(\.\d+)?$"),
            F.col("value_clean").cast("double")
        )
    )
    .drop("value_clean")
)

print("Poverty long-format rows:", poverty_long_df.count())

display(
    poverty_long_df.orderBy(
        "geography_type",
        "geography_name",
        "metric_code",
        "period_start_year",
        "definition_variant"
    )
)

In [0]:
from pyspark.sql import functions as F

poverty_validation_df = (
    poverty_long_df
    .groupBy("geography_type")
    .agg(
        F.countDistinct("geography_name").alias("geography_count"),
        F.count("*").alias("row_count"),
        F.countDistinct("metric_code").alias("metric_count"),
        F.countDistinct("survey_period").alias("survey_period_count"),
        F.sum(
            F.when(F.col("value_numeric").isNull(), 1).otherwise(0)
        ).alias("null_numeric_count")
    )
    .orderBy("geography_type")
)

duplicate_count = (
    poverty_long_df
    .groupBy(
        "geography_type",
        "geography_name",
        "metric_code",
        "survey_period",
        "definition_variant"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total rows:", poverty_long_df.count())
print("Distinct metrics:", poverty_long_df.select("metric_code").distinct().count())
print("Distinct survey periods:", poverty_long_df.select("survey_period").distinct().count())
print("Duplicate analytical records:", duplicate_count)

display(poverty_validation_df)

display(
    poverty_long_df
    .filter(F.col("value_numeric").isNull())
    .select(
        "geography_type",
        "geography_name",
        "metric_code",
        "survey_period",
        "definition_variant",
        "value_raw"
    )
)

In [0]:
silver_poverty_table = (
    "workspace.census360.silver_cbsl_poverty_indicators"
)

table_exists = spark.catalog.tableExists(silver_poverty_table)

print("Target table:", silver_poverty_table)
print("Already exists:", table_exists)

if table_exists:
    existing_poverty_df = spark.table(silver_poverty_table)
    print("Existing row count:", existing_poverty_df.count())
    print("Existing columns:", existing_poverty_df.columns)

In [0]:
from pyspark.sql import functions as F

poverty_silver_df = (
    poverty_long_df
    .withColumn(
        "value_status",
        F.when(
            F.col("value_numeric").isNull(),
            F.lit("not_available")
        ).otherwise(F.lit("available"))
    )
    .withColumn(
        "survey_period_type",
        F.when(
            F.col("period_start_year") == F.col("period_end_year"),
            F.lit("single_year")
        ).otherwise(F.lit("multi_year"))
    )
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
    .select(
        "geography_type",
        "geography_name",
        "survey_period",
        "survey_period_type",
        "period_start_year",
        "period_end_year",
        "definition_variant",
        "metric_code",
        "metric_label",
        "value_raw",
        "value_numeric",
        "value_status",
        "source_row_number",
        "source_file",
        "source_sheet",
        "silver_processed_at"
    )
)

poverty_silver_df.createOrReplaceTempView(
    "temp_cbsl_poverty_indicators"
)

spark.sql("""
    CREATE TABLE workspace.census360.silver_cbsl_poverty_indicators
    USING DELTA
    AS
    SELECT *
    FROM temp_cbsl_poverty_indicators
""")

created_poverty_df = spark.table(
    "workspace.census360.silver_cbsl_poverty_indicators"
)

print("Silver poverty table created successfully.")
print("Rows:", created_poverty_df.count())
print("Columns:", len(created_poverty_df.columns))

In [0]:
provincial_bronze_df = spark.table(
    "workspace.census360.bronze_cbsl_provincial_socioeconomic_raw"
)

print("Row count:", provincial_bronze_df.count())
print("\nColumns:")
print(provincial_bronze_df.columns)

print("\nSchema:")
provincial_bronze_df.printSchema()

print("\nRaw provincial socioeconomic rows:")
display(
    provincial_bronze_df
    .orderBy("source_row_number")
)

In [0]:
from pyspark.sql import functions as F

province_value_columns = [
    "column_02",  # Western
    "column_03",  # Central
    "column_04",  # Southern
    "column_05",  # Northern
    "column_06",  # Eastern
    "column_07",  # North Western
    "column_08",  # North Central
    "column_09",  # Uva
    "column_10",  # Sabaragamuwa
    "column_11"   # All Island
]

provincial_classified_df = (
    provincial_bronze_df

    # Assign the year represented by each worksheet block.
    .withColumn(
        "analysis_year",
        F.when(
            F.col("source_row_number").between(4, 71),
            F.lit(2016)
        )
        .when(
            F.col("source_row_number").between(73, 140),
            F.lit(2019)
        )
    )

    # Detect whether a row contains at least one provincial value.
    .withColumn(
        "has_geography_values",
        F.coalesce(
            *[F.col(column_name) for column_name in province_value_columns]
        ).isNotNull()
    )

    # Classify titles, headers, section headings, metrics and source rows.
    .withColumn(
        "row_type",
        F.when(
            F.col("source_row_number").between(1, 2),
            F.lit("title")
        )
        .when(
            F.col("source_row_number") == 3,
            F.lit("province_header")
        )
        .when(
            F.col("source_row_number").isin(4, 73),
            F.lit("year_header")
        )
        .when(
            F.col("source_row_number").isin(72, 141),
            F.lit("blank")
        )
        .when(
            F.col("source_row_number").between(142, 143),
            F.lit("source")
        )
        .when(
            F.col("source_row_number").between(5, 71)
            & F.col("has_geography_values"),
            F.lit("metric")
        )
        .when(
            F.col("source_row_number").between(74, 140)
            & F.col("has_geography_values"),
            F.lit("metric")
        )
        .when(
            F.col("source_row_number").between(5, 71),
            F.lit("section_heading")
        )
        .when(
            F.col("source_row_number").between(74, 140),
            F.lit("section_heading")
        )
        .otherwise(F.lit("unclassified"))
    )
)

display(
    provincial_classified_df
    .orderBy("source_row_number")
    .select(
        "source_row_number",
        "analysis_year",
        "row_type",
        F.col("column_01").alias("row_label"),
        "has_geography_values"
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

metric_order_window = (
    Window
    .partitionBy("analysis_year")
    .orderBy("source_row_number")
)

provincial_metric_alignment_df = (
    provincial_classified_df
    .filter(F.col("row_type") == "metric")
    .withColumn(
        "metric_position",
        F.row_number().over(metric_order_window)
    )
    .withColumn(
        "metric_label_clean",
        F.lower(
            F.regexp_replace(
                F.regexp_replace(
                    F.trim(F.col("column_01")),
                    r"[\r\n]+",
                    " "
                ),
                r"\s+",
                " "
            )
        )
    )
    .groupBy("metric_position")
    .pivot("analysis_year", [2016, 2019])
    .agg(F.first("metric_label_clean"))
    .withColumnRenamed("2016", "metric_label_2016")
    .withColumnRenamed("2019", "metric_label_2019")
    .withColumn(
        "labels_match",
        F.col("metric_label_2016") == F.col("metric_label_2019")
    )
    .orderBy("metric_position")
)

print(
    "Total metric positions:",
    provincial_metric_alignment_df.count()
)

print(
    "Mismatched metric labels:",
    provincial_metric_alignment_df
    .filter(~F.col("labels_match"))
    .count()
)

display(
    provincial_metric_alignment_df
    .filter(~F.col("labels_match"))
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

metric_order_window = (
    Window
    .partitionBy("analysis_year")
    .orderBy("source_row_number")
)

provincial_metric_context_df = (
    provincial_classified_df
    .filter(F.col("row_type") == "metric")

    # Give matching metrics the same position in both year blocks.
    .withColumn(
        "metric_position",
        F.row_number().over(metric_order_window)
    )

    # Clean line breaks and repeated spaces from metric labels.
    .withColumn(
        "metric_label_clean",
        F.regexp_replace(
            F.regexp_replace(
                F.trim(F.col("column_01")),
                r"[\r\n]+",
                " "
            ),
            r"\s+",
            " "
        )
    )

    # Preserve the major worksheet section.
    .withColumn(
        "section_name",
        F.when(
            F.col("metric_position").between(1, 2),
            F.lit("Household Characteristics")
        )
        .when(
            F.col("metric_position").between(3, 12),
            F.lit("Population Distribution")
        )
        .when(
            F.col("metric_position").between(13, 24),
            F.lit("Income")
        )
        .when(
            F.col("metric_position").between(25, 55),
            F.lit("Expenditure")
        )
    )

    # Preserve the detailed subsection context.
    .withColumn(
        "subsection_name",
        F.when(
            F.col("metric_position").between(3, 4),
            F.lit("By Gender, %")
        )
        .when(
            F.col("metric_position").between(5, 7),
            F.lit("By Age Group, %")
        )
        .when(
            F.col("metric_position").between(8, 12),
            F.lit("By Educational Attainment, %")
        )
        .when(
            F.col("metric_position").between(13, 15),
            F.lit("Mean Income, Rs. per Month")
        )
        .when(
            F.col("metric_position").between(16, 18),
            F.lit("Median Income, Rs. per Month")
        )
        .when(
            F.col("metric_position").between(19, 21),
            F.lit("Income Share by Households, %")
        )
        .when(
            F.col("metric_position").between(22, 24),
            F.lit("Gini Coefficient, One Month Income")
        )
        .when(
            F.col("metric_position") == 25,
            F.lit("Expenditure, Rs. per Month")
        )
        .when(
            F.col("metric_position").between(26, 55),
            F.lit("Household Expenditure Share, %")
        )
    )
)

display(
    provincial_metric_context_df
    .orderBy("analysis_year", "metric_position")
    .select(
        "analysis_year",
        "metric_position",
        "section_name",
        "subsection_name",
        "metric_label_clean",
        "source_row_number"
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

provincial_metric_mappings = [
    (1,  "individuals_per_household",                         "count"),
    (2,  "income_receivers_per_household",                    "count"),
    (3,  "population_male_percent",                           "percent"),
    (4,  "population_female_percent",                         "percent"),
    (5,  "population_age_0_14_percent",                       "percent"),
    (6,  "population_age_15_59_percent",                      "percent"),
    (7,  "population_age_over_59_percent",                    "percent"),
    (8,  "education_no_schooling_percent",                    "percent"),
    (9,  "education_up_to_grade_5_percent",                   "percent"),
    (10, "education_grade_6_10_percent",                      "percent"),
    (11, "education_passed_gce_ol_percent",                   "percent"),
    (12, "education_passed_gce_al_and_above_percent",         "percent"),

    (13, "mean_income_per_household_monthly_lkr",             "LKR_per_month"),
    (14, "mean_income_per_person_monthly_lkr",                "LKR_per_month"),
    (15, "mean_income_per_income_receiver_monthly_lkr",       "LKR_per_month"),
    (16, "median_income_per_household_monthly_lkr",           "LKR_per_month"),
    (17, "median_income_per_person_monthly_lkr",              "LKR_per_month"),
    (18, "median_income_per_income_receiver_monthly_lkr",     "LKR_per_month"),

    (19, "income_share_richest_20_percent",                   "percent"),
    (20, "income_share_poorest_20_percent",                   "percent"),
    (21, "income_share_middle_60_percent",                    "percent"),

    (22, "gini_coefficient_households",                       "coefficient"),
    (23, "gini_coefficient_per_person",                       "coefficient"),
    (24, "gini_coefficient_income_receivers",                 "coefficient"),

    (25, "household_expenditure_monthly_lkr",                 "LKR_per_month"),

    (26, "expenditure_share_all_food_percent",                "percent"),
    (27, "expenditure_share_cereal_percent",                  "percent"),
    (28, "expenditure_share_prepared_foods_percent",          "percent"),
    (29, "expenditure_share_pulses_percent",                  "percent"),
    (30, "expenditure_share_vegetables_percent",              "percent"),
    (31, "expenditure_share_meat_percent",                    "percent"),
    (32, "expenditure_share_fish_percent",                    "percent"),
    (33, "expenditure_share_dried_fish_percent",              "percent"),
    (34, "expenditure_share_eggs_percent",                    "percent"),
    (35, "expenditure_share_coconut_percent",                 "percent"),
    (36, "expenditure_share_condiments_percent",              "percent"),
    (37, "expenditure_share_milk_products_percent",           "percent"),
    (38, "expenditure_share_fat_and_oil_percent",             "percent"),
    (39, "expenditure_share_sugar_jaggery_treacle_percent",   "percent"),
    (40, "expenditure_share_fruits_percent",                  "percent"),
    (41, "expenditure_share_other_food_percent",              "percent"),

    (42, "expenditure_share_all_non_food_percent",            "percent"),
    (43, "expenditure_share_housing_percent",                 "percent"),
    (44, "expenditure_share_fuel_and_light_percent",          "percent"),
    (45, "expenditure_share_personal_care_health_percent",    "percent"),
    (46, "expenditure_share_transport_percent",               "percent"),
    (47, "expenditure_share_communication_percent",           "percent"),
    (48, "expenditure_share_education_percent",               "percent"),
    (49, "expenditure_share_culture_entertainment_percent",   "percent"),
    (50, "expenditure_share_non_durable_household_percent",   "percent"),
    (51, "expenditure_share_clothing_footwear_percent",       "percent"),
    (52, "expenditure_share_durable_household_goods_percent", "percent"),
    (53, "expenditure_share_miscellaneous_percent",           "percent"),
    (54, "expenditure_share_rare_expenses_percent",           "percent"),
    (55, "expenditure_share_liquor_tobacco_percent",          "percent")
]

mapping_schema = StructType([
    StructField("metric_position", IntegerType(), False),
    StructField("metric_code", StringType(), False),
    StructField("unit", StringType(), False)
])

provincial_metric_mapping_df = spark.createDataFrame(
    provincial_metric_mappings,
    schema=mapping_schema
)

provincial_metrics_coded_df = (
    provincial_metric_context_df
    .join(
        provincial_metric_mapping_df,
        on="metric_position",
        how="left"
    )
)

print("Total rows:", provincial_metrics_coded_df.count())

print(
    "Distinct metric codes:",
    provincial_metrics_coded_df
    .select("metric_code")
    .distinct()
    .count()
)

print(
    "Rows with missing metric codes:",
    provincial_metrics_coded_df
    .filter(F.col("metric_code").isNull())
    .count()
)

display(
    provincial_metrics_coded_df
    .orderBy("analysis_year", "metric_position")
    .select(
        "analysis_year",
        "metric_position",
        "section_name",
        "subsection_name",
        "metric_code",
        "metric_label_clean",
        "unit"
    )
)

In [0]:
from pyspark.sql import functions as F

province_mappings = [
    ("Western", "province", "column_02"),
    ("Central", "province", "column_03"),
    ("Southern", "province", "column_04"),
    ("Northern", "province", "column_05"),
    ("Eastern", "province", "column_06"),
    ("North Western", "province", "column_07"),
    ("North Central", "province", "column_08"),
    ("Uva", "province", "column_09"),
    ("Sabaragamuwa", "province", "column_10"),
    ("All Island", "national", "column_11")
]

province_structs = [
    F.struct(
        F.lit(geography_name).alias("geography_name"),
        F.lit(geography_type).alias("geography_type"),
        F.col(source_column).alias("value_raw")
    )
    for geography_name, geography_type, source_column in province_mappings
]

provincial_long_df = (
    provincial_metrics_coded_df
    .withColumn(
        "geography_values",
        F.array(*province_structs)
    )
    .withColumn(
        "geography_value",
        F.explode("geography_values")
    )
    .select(
        "analysis_year",
        "metric_position",
        "section_name",
        "subsection_name",
        "metric_code",
        F.col("metric_label_clean").alias("metric_label"),
        "unit",
        F.col("geography_value.geography_type").alias("geography_type"),
        F.col("geography_value.geography_name").alias("geography_name"),
        F.col("geography_value.value_raw").alias("value_raw"),
        "source_row_number",
        "source_file",
        "source_sheet"
    )
    .withColumn(
        "value_clean",
        F.regexp_replace(
            F.trim(F.col("value_raw")),
            ",",
            ""
        )
    )
    .withColumn(
        "value_numeric",
        F.when(
            F.col("value_clean").rlike(r"^-?\d+(\.\d+)?$"),
            F.col("value_clean").cast("double")
        )
    )
    .drop("value_clean")
)

print("Provincial long-format rows:", provincial_long_df.count())

display(
    provincial_long_df.orderBy(
        "analysis_year",
        "metric_position",
        "geography_type",
        "geography_name"
    )
)

In [0]:
from pyspark.sql import functions as F

provincial_validation_df = (
    provincial_long_df
    .groupBy("analysis_year", "geography_type")
    .agg(
        F.countDistinct("geography_name").alias("geography_count"),
        F.countDistinct("metric_code").alias("metric_count"),
        F.count("*").alias("row_count"),
        F.sum(
            F.when(F.col("value_numeric").isNull(), 1).otherwise(0)
        ).alias("null_numeric_count")
    )
    .orderBy("analysis_year", "geography_type")
)

duplicate_count = (
    provincial_long_df
    .groupBy(
        "analysis_year",
        "geography_type",
        "geography_name",
        "metric_code"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total rows:", provincial_long_df.count())
print(
    "Distinct years:",
    provincial_long_df.select("analysis_year").distinct().count()
)
print(
    "Distinct metrics:",
    provincial_long_df.select("metric_code").distinct().count()
)
print(
    "Distinct geographies:",
    provincial_long_df.select("geography_name").distinct().count()
)
print("Duplicate analytical records:", duplicate_count)

display(provincial_validation_df)

In [0]:
silver_provincial_table = (
    "workspace.census360.silver_cbsl_provincial_socioeconomic"
)

table_exists = spark.catalog.tableExists(silver_provincial_table)

print("Target table:", silver_provincial_table)
print("Already exists:", table_exists)

if table_exists:
    existing_provincial_df = spark.table(silver_provincial_table)
    print("Existing row count:", existing_provincial_df.count())
    print("Existing columns:", existing_provincial_df.columns)

In [0]:
from pyspark.sql import functions as F

provincial_silver_df = (
    provincial_long_df
    .withColumn(
        "value_status",
        F.when(
            F.col("value_numeric").isNull(),
            F.lit("not_available")
        ).otherwise(F.lit("available"))
    )
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
    .select(
        "analysis_year",
        "geography_type",
        "geography_name",
        "section_name",
        "subsection_name",
        "metric_position",
        "metric_code",
        "metric_label",
        "unit",
        "value_raw",
        "value_numeric",
        "value_status",
        "source_row_number",
        "source_file",
        "source_sheet",
        "silver_processed_at"
    )
)

provincial_silver_df.createOrReplaceTempView(
    "temp_cbsl_provincial_socioeconomic"
)

spark.sql("""
    CREATE TABLE workspace.census360.silver_cbsl_provincial_socioeconomic
    USING DELTA
    AS
    SELECT *
    FROM temp_cbsl_provincial_socioeconomic
""")

created_provincial_df = spark.table(
    "workspace.census360.silver_cbsl_provincial_socioeconomic"
)

print("Silver provincial socioeconomic table created successfully.")
print("Rows:", created_provincial_df.count())
print("Columns:", len(created_provincial_df.columns))

In [0]:
silver_tables = [
    "workspace.census360.silver_cbsl_telecommunication_services",
    "workspace.census360.silver_cbsl_household_income_hies",
    "workspace.census360.silver_cbsl_poverty_indicators",
    "workspace.census360.silver_cbsl_provincial_socioeconomic"
]

for table_name in silver_tables:
    table_exists = spark.catalog.tableExists(table_name)

    print("\nTable:", table_name)
    print("Exists:", table_exists)

    if table_exists:
        table_df = spark.table(table_name)

        print("Rows:", table_df.count())
        print("Columns:", len(table_df.columns))

In [0]:
trcsl_telecom_df = spark.table(
    "workspace.census360.silver_trcsl_telecom_statistics"
)

print("Row count:", trcsl_telecom_df.count())

print("\nColumns:")
print(trcsl_telecom_df.columns)

print("\nSchema:")
trcsl_telecom_df.printSchema()

print("\nTRCSL telecom data:")
display(trcsl_telecom_df)